# Import Library

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack
from collections import defaultdict
from difflib import get_close_matches
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import coo_matrix

# UI using tkinter
import sys
from PyQt5.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout, 
                             QHBoxLayout, QLineEdit, QPushButton, QListWidget, 
                             QLabel, QStackedWidget, QComboBox, QSpinBox, 
                             QDoubleSpinBox, QMessageBox, QListWidgetItem)
from PyQt5.QtCore import Qt
from PyQt5.QtGui import QFont, QPixmap
from PyQt5.QtMultimedia import QMediaPlayer, QMediaContent
from PyQt5.QtCore import QUrl

# Content-Based Filtering

In [2]:
df = pd.read_csv('data/newMusicInfo.csv')
df.head()

,track_id,name,artist,spotify_preview_url,tags,genre,year,duration_ms,danceability,speechiness,tempo
0,TROUVHL128F426C441,Come as You Are,NIRVANA,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, GRUNGE",RNB,1991,218920,0.508,0.0400,120.012
1,TRLNZBD128F935E4D8,Creep,RADIOHEAD,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...,"ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...",RNB,2008,238640,0.515,0.0369,91.841
2,TRMZXEW128F9341FD5,The Scientist,COLDPLAY,https://p.scdn.co/mp3-preview/95cb9df1b056d759...,"ROCK, ALTERNATIVE, INDIE, POP, ALTERNATIVE_ROC...",ROCK,2007,311014,0.566,0.0242,146.365
3,TRIODZU128E078F3E2,Under the Bridge,RED HOT CHILI PEPPERS,https://p.scdn.co/mp3-preview/90e41778392f27b6...,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, FUNK",POP,2003,265506,0.554,0.0457,84.275
4,TRYEZGE128F92D04C8,Feel Good Inc.,GORILLAZ,https://p.scdn.co/mp3-preview/7388c425022ced92...,"ROCK, ELECTRONIC, ALTERNATIVE, INDIE, POP, ALT...",RNB,2010,223106,0.809,0.1360,138.556


### Selected only the relevant features

In [3]:
df_selected = df[['track_id', 'name','artist', 'tags', 'genre', 'year', 'danceability', 'speechiness', 'tempo']]
df_selected.head()

,track_id,name,artist,tags,genre,year,danceability,speechiness,tempo
0,TROUVHL128F426C441,Come as You Are,NIRVANA,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, GRUNGE",RNB,1991,0.508,0.0400,120.012
1,TRLNZBD128F935E4D8,Creep,RADIOHEAD,"ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...",RNB,2008,0.515,0.0369,91.841
2,TRMZXEW128F9341FD5,The Scientist,COLDPLAY,"ROCK, ALTERNATIVE, INDIE, POP, ALTERNATIVE_ROC...",ROCK,2007,0.566,0.0242,146.365
3,TRIODZU128E078F3E2,Under the Bridge,RED HOT CHILI PEPPERS,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, FUNK",POP,2003,0.554,0.0457,84.275
4,TRYEZGE128F92D04C8,Feel Good Inc.,GORILLAZ,"ROCK, ELECTRONIC, ALTERNATIVE, INDIE, POP, ALT...",RNB,2010,0.809,0.1360,138.556


### Check data type

In [4]:
df_selected.dtypes

track_id         object
name             object
artist           object
tags             object
genre            object
year              int64
danceability    float64
speechiness     float64
tempo           float64
dtype: object

### Copy and clean

In [5]:
df_selected = df[['track_id', 'name', 'artist', 'tags', 'genre', 'year', 'danceability', 'speechiness', 'tempo']].copy()
df_selected[['name', 'artist', 'tags', 'genre']] = df_selected[['name', 'artist', 'tags', 'genre']].fillna('')

### Combine text features

In [6]:
df_selected['combined_text'] = df_selected['tags'] + ' ' + df_selected['genre'] + ' ' + df_selected['artist']

### TF-IDF on combined text

In [7]:
# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(
    analyzer='word', 
    ngram_range=(1, 2),  # Include unigrams and bigrams
    min_df=1, # Keep words that appear in at least 1 document
    stop_words='english'
)

# Transform the combined text column into TF-IDF features
text_matrix = tfidf.fit_transform(df_selected['combined_text'])

### Normalize numeric features

In [8]:
# Initialize and apply MinMaxScaler to scale all numeric columns to [0, 1]
scaler = MinMaxScaler()
# Apply scaling to numerical features
numeric_matrix = scaler.fit_transform(df_selected[['danceability', 'speechiness', 'tempo', 'year']])

### Combine all features

In [9]:
combined_matrix = hstack([text_matrix, numeric_matrix]).tocsr()

### Fit KNN model

In [10]:
knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
knn_model.fit(combined_matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

### Build name, tag, genre, year and index mapping

In [11]:
# lowercase song names in mapping
# Build the (song name) -> index map
name_to_index = {
    row['name'].lower().strip(): idx
    for idx, row in df_selected.iterrows()
}

In [12]:
# Create a dictionary to map (tag, genre, year) -> list of row indices
tag_genre_year_to_indices = {}

# Loop through each row in the dataframe
for idx, row in df_selected.iterrows():
    # Build a unique key using tag, genre, and year (all lowercase for consistency)
    key = (row['tags'].lower().strip(), row['genre'].lower().strip(), row['year'])
    
    # Insert the key into the dictionary if it doesn't exist, and append the current row index to its list
    tag_genre_year_to_indices.setdefault(key, []).append(idx)

In [13]:
# Create a dictionary to map (genre, year) -> list of row indices
genre_year_to_indices = {}

# Loop through each row in the dataframe
for idx, row in df_selected.iterrows():
    # Build a unique key using genre and year (all lowercase for consistency)
    key = (row['genre'].lower().strip(), row['year'])
    
     # Insert the key into the dictionary if it doesn't exist, and append the current row index to its list
    genre_year_to_indices.setdefault(key, []).append(idx)

In [14]:
genre_to_indices = {}

# Loop through each row in the dataframe
for idx, row in df_selected.iterrows():
    # Build a unique key using genre (all lowercase for consistency)
    key = str(row['genre']).lower().strip()
    
     # Insert the key into the dictionary if it doesn't exist, and append the current row index to its list
    genre_to_indices.setdefault(key, []).append(idx)

### Recommendation function using name

##### Calculation accuracy function

In [15]:
def calculate_accuracy_from_distance(distances):
    # Convert KNN distances into similarity percentages.
    return [(1 - d) * 100 for d in distances]

##### Validation for tempo, danceability and year

In [16]:
def validate_song_name(song_name):
    # Validate that the song exists in the dataset
    song_key = song_name.lower().strip()
    if song_key not in name_to_index:
        return None, f"Song '{song_name}' not found in the dataset."
    return song_key, None

##### Main Recommendation using name Function

In [17]:
def recommend_songs_by_name(song_name, n_recommendations=5):
    # Preprocess the input name (normalize: lowercase + remove spaces)
    song_key = song_name.lower().strip()
    
    # 1 - Check if the song exists in the mapping
    song_key, error = validate_song_name(song_name)
    if error:
        return error
    
    # 2 - Get the index of the input song directly from the dictionary
    idx = name_to_index[song_key] 
    
    # 3 - Find nearest neighbors using the KNN model
    distances, indices = knn_model.kneighbors(
        combined_matrix[idx], n_neighbors=n_recommendations + 1
    )
    
    # 4 - Exclude the first result (it’s always the song itself)
    recommended_indices = indices.flatten()[1:]
    recommended_distances = distances.flatten()[1:]
    
    # 5 - Get details of the input song
    input_song = df_selected.iloc[[idx]][
        ['track_id','name','artist','year','genre','tempo','tags','danceability']
    ]
    
    # 6 - Get details of recommended songs
    recommendations = df_selected.iloc[recommended_indices][
        ['track_id','name','artist','year','genre','tempo','tags','danceability']
    ]
    
    # 7 - Add similarity score ("accuracy") column and calculation 
    input_song = input_song.assign(accuracy=["100%"])
    similarities = calculate_accuracy_from_distance(recommended_distances)
    recommendations = recommendations.assign(
        accuracy=[f"{s:.2f}%" for s in similarities]
    )
    
    # 8 - Merge input + recommendations into one DataFrame
    return pd.concat([input_song, recommendations]).reset_index(drop=True)

### Recommendation function using genre, tags, tempo and danceability

##### Calculation accuracy function

In [18]:
def calculate_accuracy(song_tempo, song_dance, input_tempo, input_dance):
    
    # Calculate similarity with input manually (tempo + danceability diff)
    tempo_diff = abs(song_tempo - input_tempo) / max(input_tempo, 1)
    dance_diff = abs(song_dance - input_dance)
    
    # Simple similarity formula: 1 - take 50% weight from tempo difference + 50% from danceability difference.
    sim = (1 - (0.5 * tempo_diff + 0.5 * dance_diff)) * 100
    
    # Clip the score between 0 and 100 to avoid negative or >100 values.
    sim = max(0, min(100, sim)) 
    
    return f"{sim:.2f}%"

##### Validation for tempo, danceability and year

In [19]:
def validate_inputs(song_tempo, danceability_value, year):
    
    # Validate tempo (must be numeric)
    try:
        song_tempo = float(song_tempo)
    except ValueError:
        return "Tempo must be a number (BPM)."

    # Validate danceability (0.0–1.0)
    try:
        danceability_value = float(danceability_value)
    except ValueError:
        return "Danceability must be between 0.0 and 1.0."
    if not (0.0 <= danceability_value <= 1.0):
        return "Danceability must be between 0.0 and 1.0."

    # Validate year (must be integer)
    try:
        year = int(year)
    except ValueError:
        return "Year must be an integer."

    return song_tempo, danceability_value, year


##### Main Recommendation using genre, tags, tempo and danceability Function

In [20]:
def recommend_songs_by_genre_tempo_tags_danceability(
    song_genre, song_tempo, song_tags, danceability_value, year, n_recommendations=5
):
    # Normalize text fields (remove spaces + lowercase for consistency)
    song_genre = song_genre.lower().strip()
    song_tags = song_tags.lower().strip()

    # Run input validation
    validated = validate_inputs(song_tempo, danceability_value, year)
    
    # error message returned
    if isinstance(validated, str):  
        return validated 
    song_tempo, danceability_value, year = validated

    # 1 - fallback search
    candidates = None
    key_tag = (song_tags, song_genre, year)
    
    # First try - (tag + genre + year) exact match
    if key_tag in tag_genre_year_to_indices:
        candidates = df_selected.iloc[tag_genre_year_to_indices[key_tag]]

    # Second try - (genre + year)
    elif (song_genre, year) in genre_year_to_indices:
        candidates = df_selected.iloc[genre_year_to_indices[(song_genre, year)]]

    # Third try - only genre
    elif song_genre in genre_to_indices:
        candidates = df_selected.iloc[genre_to_indices[song_genre]]

    # If nothing matches - return not found message
    if candidates is None or candidates.empty:
        return f"No songs found in genre '{song_genre}' for year {year} (with/without tag '{song_tags}')."

    # 2 - Tempo filter ±5
    tempo_matches = candidates[(candidates['tempo'] >= song_tempo - 5) & (candidates['tempo'] <= song_tempo + 5)]
    if tempo_matches.empty:
        return f"No songs in genre '{song_genre}' near {song_tempo} BPM."

    # 3 - Danceability filter ±0.1
    final_matches = tempo_matches[
        (tempo_matches['danceability'] >= danceability_value - 0.1) & 
        (tempo_matches['danceability'] <= danceability_value + 0.1)
    ].copy()
    if final_matches.empty:
        return f"No songs in genre '{song_genre}' near {song_tempo} BPM with danceability {danceability_value}."

    # 4 - Pick the "closest" candidate (min absolute difference in danceability)
    final_matches['diff'] = (
        (final_matches['tempo'] - song_tempo).abs() +
        (final_matches['danceability'] - danceability_value).abs() + 
        (final_matches['year'] - year).abs()
    )
    idx = final_matches['diff'].idxmin()

    # 5 - Run KNN
    distances,indices = knn_model.kneighbors(combined_matrix[idx], n_neighbors=n_recommendations + 1)
    recommended_indices = indices.flatten()[1:]
    # recommended_distances = distances.flatten()[1:]

    # 6 - Get details of the input song
    input_song = df_selected.iloc[[idx]][['track_id','name','artist','year','genre','tempo','tags','danceability']]
    
    # Check if input exactly matches the chosen candidate (strict match on 4 fields)
    is_exact_match = (
        input_song['genre'].iloc[0].lower().strip() == song_genre and
        input_song['tags'].iloc[0].lower().strip() == song_tags and
        input_song['year'].iloc[0] == year and
        abs(input_song['tempo'].iloc[0] - song_tempo) < 1e-6 and
        abs(input_song['danceability'].iloc[0] - danceability_value) < 1e-6
        
    )
    
    # 7 - Get details of recommended songs
    recommendations = df_selected.iloc[recommended_indices][['track_id','name','artist','year','genre','tempo','tags','danceability']]

    # 8 - Add similarity score ("accuracy") column and calculation 
    if is_exact_match:
        input_song = input_song.assign(accuracy=["100%"])
    else:
        ## Input song accuracy     
        acc = calculate_accuracy(
            input_song['tempo'].iloc[0],
            input_song['danceability'].iloc[0],
            song_tempo,
            danceability_value
        )
        input_song = input_song.assign(accuracy=[acc])
        
    ## Recommended songs accuracy
    # For each recommended song, calculate similarity to the user’s input values.
    recommendations = df_selected.iloc[recommended_indices][['track_id','name','artist','year','genre','tempo','tags','danceability']].copy()
    recommendations['accuracy_val'] = recommendations.apply(
        lambda row: float(calculate_accuracy(row['tempo'], row['danceability'], song_tempo, danceability_value).replace("%","")),
        axis=1
    )
    
    # Sort by accuracy high → low
    recommendations = recommendations.sort_values(by="accuracy_val", ascending=False)
    
    # Format accuracy
    recommendations['accuracy'] = recommendations['accuracy_val'].map(lambda s: f"{s:.2f}%")
    recommendations = recommendations.drop(columns="accuracy_val")

    # 9 - Merge input + recommendations into one DataFrame
    return pd.concat([input_song, recommendations]).reset_index(drop=True)


# Collaborative Filtering System

## Then load the datasets 

In [21]:
user_history = pd.read_csv('data/newUserListeningHistory.csv')
music_info = pd.read_csv('data/newMusicInfo.csv')

## encode user and track id

In [22]:
user_encoder = LabelEncoder()
track_encoder = LabelEncoder()
user_history['user_index'] = user_encoder.fit_transform(user_history['user_id'])
user_history['track_index'] = track_encoder.fit_transform(user_history['track_id'])

## create sparse item matrix

In [23]:
rows = user_history['user_index']
cols = user_history['track_index']
data = user_history['playcount'].astype(np.float32)
num_users = rows.nunique()
num_items = cols.nunique()
user_item_matrix = coo_matrix((data, (rows, cols)), shape=(num_users, num_items)).tocsr()

## apply truncated SVD

In [24]:
svd = TruncatedSVD(n_components=20, random_state=42)
svd_matrix = svd.fit_transform(user_item_matrix)

## convert index to id to map them

In [25]:
track_id_to_info = music_info.set_index('track_id')[['name', 'artist']].to_dict('index')
user_index_to_id = dict(enumerate(user_encoder.classes_))
track_index_to_id = dict(enumerate(track_encoder.classes_))

## recommend based on user ID then compare for similarity then suggest song that havent exist in the album of the user input id

In [26]:
def recommend_by_user_id(user_id_input, top_n=5):
    if user_id_input not in user_encoder.classes_:
        suggestions = get_close_matches(user_id_input, user_encoder.classes_, n=3)
        return f"User ID not found. Did you mean: {', '.join(suggestions)}?"

    user_index = user_encoder.transform([user_id_input])[0]
    target_vector = svd_matrix[user_index]
    
    # Calculate cosine similarity with ALL users
    similarities = cosine_similarity([target_vector], svd_matrix)[0]

    # Get top 5 similar users (excluding self)
    top_user_indices = np.argsort(-similarities)[1:6]
    
    print(f"\nInput User ID: {user_id_input}")
    print("Top 5 similar users and their similarity scores:")
    user_similarities = {}
    for i, user_idx in enumerate(top_user_indices, 1):
        sim_id = user_index_to_id[user_idx]
        sim_score = similarities[user_idx]
        user_similarities[user_idx] = sim_score
        print(f" {i}. {sim_id} (Similarity: {sim_score:.4f})")

    print(f"\nAnalyzing songs from these {len(top_user_indices)} similar users...")
    
    # Track which users listened to which songs
    track_to_user_similarities = defaultdict(list)
    
    for sim_idx in top_user_indices:
        uid = user_index_to_id[sim_idx]
        sim_score = similarities[sim_idx]
        user_tracks = user_history[user_history['user_index'] == sim_idx]['track_index'].unique()
        
        for tidx in user_tracks:
            track_to_user_similarities[tidx].append({
                'user_id': uid,
                'user_index': sim_idx,
                'similarity': sim_score
            })

    # Get songs user has already listened to
    listened = set(user_history[user_history['user_index'] == user_index]['track_index'].unique())
    
    # Score tracks based on weighted average of similar users' preferences
    track_scores = {}
    for track_idx, user_list in track_to_user_similarities.items():
        if track_idx not in listened:  # Only recommend unheard songs
            # Weight by similarity scores
            weighted_score = sum(user_info['similarity'] for user_info in user_list)
            track_scores[track_idx] = {
                'score': weighted_score,
                'contributing_users': user_list
            }
    
    # Sort by weighted score and get top N
    top_tracks = sorted(track_scores.items(), key=lambda x: x[1]['score'], reverse=True)[:top_n]
    
    recommendations = []
    for track_idx, track_data in top_tracks:
        track_id = track_index_to_id[track_idx]
        info = track_id_to_info.get(track_id, {'name': 'Unknown', 'artist': 'Unknown'})
        
        # Format contributing users with their similarities
        contributing_users = []
        for user_info in track_data['contributing_users']:
            contributing_users.append(f"{user_info['user_id']} ({user_info['similarity']:.4f})")

         # Get Spotify URL and duration from original data
        track_row = df[df['track_id'] == track_id]
        spotify_url = track_row['spotify_preview_url'].values[0] if not track_row.empty else 'Unknown'
        duration_ms = track_row['duration_ms'].values[0] if not track_row.empty else 0
        
        # Convert duration from ms to mm:ss format
        duration_formatted = f"{int(duration_ms)//60000}:{(int(duration_ms)%60000)//1000:02d}" if duration_ms else '0:00'
        
        recommendations.append({
            'track_id': track_id,
            'name': info['name'],
            'artist': info['artist'],
            'weighted_score': track_data['score'],
            'num_similar_users': len(track_data['contributing_users']),
            'contributing_users_with_similarity': '; '.join(contributing_users),  
            'spotify_preview_url': spotify_url,  
            'duration': duration_formatted  
        })
    
    return pd.DataFrame(recommendations)

# Hybrid Recommendation System

In [27]:
# Hybrid recommendation function (combines content-based and collaborative filtering)
def hybrid_recommendation(song_name, user_id, n_recommendations=5):
    """
    Hybrid recommendation that combines:
    1. Content-based filtering based on the searched song
    2. Collaborative filtering based on user preferences
    """
    # Get content-based recommendations
    content_recs = recommend_songs_by_name(song_name, n_recommendations * 2)
    
    # Get collaborative recommendations
    collab_recs = recommend_by_user_id(user_id, n_recommendations * 2)
    
    # Combine and deduplicate (simple hybrid approach)
    # In a real implementation, you would use a more sophisticated blending method
    hybrid_results = pd.concat([content_recs, collab_recs]).drop_duplicates(subset=['track_id'])
    
    # Return top n recommendations
    return hybrid_results.head(n_recommendations)

# UI and Hybrid Recommendation System

In [ ]:
# Mock recommendation functions with spotify_preview_url
def recommend_songs_by_name(song_name, n_recommendations=5):
    return pd.DataFrame({
        'name': [f'Song {i}' for i in range(1, n_recommendations+1)],
        'artist': [f'Artist {i}' for i in range(1, n_recommendations+1)],
        'track_id': [f'track_{i}' for i in range(1, n_recommendations+1)],
        'spotify_preview_url': [f'https://www.soundjay.com/misc/sounds/bell-ringing-05.mp3' for i in range(1, n_recommendations+1)],
        'duration': ['0:30'] * n_recommendations
    })

def recommend_songs_by_genre_tempo_tags_danceability(song_genre, song_tempo, song_tags, danceability_value, year, n_recommendations=5):
    return pd.DataFrame({
        'name': [f'Filtered Song {i}' for i in range(1, n_recommendations+1)],
        'artist': [f'Filtered Artist {i}' for i in range(1, n_recommendations+1)],
        'track_id': [f'filtered_track_{i}' for i in range(1, n_recommendations+1)],
        'spotify_preview_url': [f'https://www.soundjay.com/misc/sounds/bell-ringing-05.mp3' for i in range(1, n_recommendations+1)],
        'duration': ['0:30'] * n_recommendations
    })

def recommend_by_user_id(user_id, top_n=5):
    return pd.DataFrame({
        'name': [f'User Song {i}' for i in range(1, top_n+1)],
        'artist': [f'User Artist {i}' for i in range(1, top_n+1)],
        'track_id': [f'user_track_{i}' for i in range(1, top_n+1)],
        'weighted_score': [0.9 - i*0.1 for i in range(top_n)],
        'num_similar_users': [3, 2, 4, 1, 2],
        'contributing_users_with_similarity': ['user123 (0.85); user456 (0.78)'] * top_n,
        'spotify_preview_url': [f'https://www.soundjay.com/misc/sounds/bell-ringing-05.mp3' for i in range(1, top_n+1)],
        'duration': ['0:30'] * top_n
    })

def hybrid_recommendation(song_name, user_id, n_recommendations=5):
    content_recs = recommend_songs_by_name(song_name, n_recommendations * 2)
    collab_recs = recommend_by_user_id(user_id, n_recommendations * 2)
    hybrid_results = pd.concat([content_recs, collab_recs]).drop_duplicates(subset=['track_id'])
    return hybrid_results.head(n_recommendations)

# Custom widget for song item with image, play button, and audio
class SongWidget(QWidget):
    def __init__(self, name, artist, preview_url, duration, parent=None):
        super().__init__(parent)
        self.preview_url = preview_url
        self.is_playing = False
        self.media_player = QMediaPlayer()
        
        layout = QHBoxLayout()
        
        # Disc image (placeholder)
        disc_label = QLabel()
        disc_label.setFixedSize(40, 40)
        disc_label.setStyleSheet("background-color: #ccc; border-radius: 20px;")
        disc_label.setAlignment(Qt.AlignCenter)
        disc_label.setText("🎵")
        layout.addWidget(disc_label)
        
        # Song info
        info_layout = QVBoxLayout()
        name_label = QLabel(name)
        name_label.setStyleSheet("font-weight: bold;")
        artist_label = QLabel(artist)
        info_layout.addWidget(name_label)
        info_layout.addWidget(artist_label)
        layout.addLayout(info_layout)
        
        # Play button
        self.play_btn = QPushButton("▶")
        self.play_btn.setFixedSize(30, 30)
        self.play_btn.clicked.connect(self.toggle_play)
        layout.addWidget(self.play_btn)
        
        # Duration
        duration_label = QLabel(duration)
        layout.addWidget(duration_label)
        
        self.setLayout(layout)
    
    def toggle_play(self):
        if self.is_playing:
            self.media_player.stop()
            self.play_btn.setText("▶")
            self.is_playing = False
        else:
            if self.preview_url:
                self.media_player.setMedia(QMediaContent(QUrl(self.preview_url)))
                self.media_player.play()
                self.play_btn.setText("⏹")
                self.is_playing = True

# Login Page
class LoginPage(QWidget):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.parent = parent
        self.initUI()
        
    def initUI(self):
        layout = QVBoxLayout()
        
        title = QLabel("Login")
        title.setFont(QFont("Arial", 18, QFont.Bold))
        title.setAlignment(Qt.AlignCenter)
        layout.addWidget(title)
        
        layout.addSpacing(30)
        
        self.user_id_input = QLineEdit()
        self.user_id_input.setPlaceholderText("Enter User ID")
        layout.addWidget(self.user_id_input)
        
        login_btn = QPushButton("Login")
        login_btn.clicked.connect(self.login)
        layout.addWidget(login_btn)
        
        back_btn = QPushButton("Back to Home")
        back_btn.clicked.connect(lambda: self.parent.stacked_widget.setCurrentIndex(0))
        layout.addWidget(back_btn)
        
        self.setLayout(layout)
    
    def login(self):
        user_id = self.user_id_input.text().strip()
        if user_id:
            self.parent.current_user = user_id
            self.parent.update_recommendations()
            self.parent.stacked_widget.setCurrentIndex(0)  # Go back to home
            QMessageBox.information(self, "Success", f"Logged in as user {user_id}")
        else:
            QMessageBox.warning(self, "Error", "Please enter a valid User ID")

# Filter Page
class FilterPage(QWidget):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.parent = parent
        self.initUI()
        
    def initUI(self):
        layout = QVBoxLayout()
        
        title = QLabel("Filter Songs")
        title.setFont(QFont("Arial", 18, QFont.Bold))
        title.setAlignment(Qt.AlignCenter)
        layout.addWidget(title)
        
        # Genre selection
        genre_label = QLabel("Genre:")
        layout.addWidget(genre_label)
        
        self.genre_combo = QComboBox()
        genres = ['RNB', 'ROCK', 'POP', 'METAL', 'ELECTRONIC', 'JAZZ', 'PUNK', 
                 'COUNTRY', 'FOLK', 'REGGAE', 'RAP', 'BLUES', 'NEW AGE', 'LATIN', 'WORLD']
        self.genre_combo.addItems(genres)
        layout.addWidget(self.genre_combo)
        
        # Tags selection (single selection like genre)
        tags_label = QLabel("Tags (select one):")
        layout.addWidget(tags_label)
        
        self.tags_combo = QComboBox()
        tags = ['PROGRESSIVE_METAL', 'EMO', 'DANCE',
               'PSYCHEDELIC', 'GOTHIC_METAL', 'DOWNTEMPO', 'POWER_METAL', 'SYNTHPOP', 
               'DEATH_METAL', 'PROGRESSIVE_ROCK', 'THRASH_METAL', 
               'SINGER_SONGWRITER', 'CLASSICAL', 'PIANO', 'CLASSIC_ROCK', 
               'INDIE_ROCK', 'INDIE', 'SOUNDTRACK', 'LOUNGE', 'BLACK_METAL', 
               'AVANT_GARDE', 'J_POP', 'HEAVY_METAL', 'POST_ROCK', 'SKA', 'HARDCORE', 
               'GUITAR', 'HIP_HOP', 'SYMPHONIC_METAL', 'NU_METAL', 
               'HARD_ROCK', 'POST_PUNK', 'INDUSTRIAL', 'GRUNGE', 'AMBIENT', 'TRIP_HOP', 
               'BRITISH', 'JAPANESE', 'METALCORE', 'INSTRUMENTAL', 'NEW_WAVE', 'FRENCH', 
               'AMERICAN', 'TECHNO', 'GOTHIC', 'ROCK', 'NOISE', 'BLUES', 'ELECTRO', 
               'SCREAMO', 'POP_ROCK', 'HOUSE', 'DOOM_METAL', 'SOUL', 'CHILL', 'BRITPOP', 
               'BLUES_ROCK', 'RUSSIAN', 'DRUM_AND_BASS', 'GRINDCORE', 'ELECTRONIC', 
               'POST_HARDCORE', 'PSYCHEDELIC_ROCK', 'RNB', 'POLISH', 'COVER', 'PUNK_ROCK', 
               'LOVE', 'FEMALE_VOCALISTS', 'EXPERIMENTAL', 'ALTERNATIVE_ROCK', 
               'MALE_VOCALISTS', 'OLDIES', 'CHILLOUT', 'ACOUSTIC', 
               'MELODIC_DEATH_METAL', 'SWEDISH', 'MELLOW', 'ALTERNATIVE', 
               'DARK_AMBIENT', 'GERMAN', 'TRANCE', 'IDM', 'INDIE_POP']
        
        # Sort tags alphabetically
        sorted_tags = sorted(tags)
        self.tags_combo.addItem('BEAUTIFUL')  # Add empty option
        for tag in sorted_tags:
            self.tags_combo.addItem(tag)
            
        layout.addWidget(self.tags_combo)
        
        # Year input
        year_label = QLabel("Year:")
        layout.addWidget(year_label)
        
        self.year_input = QSpinBox()
        self.year_input.setRange(1900, 2030)
        self.year_input.setValue(2020)
        layout.addWidget(self.year_input)
        
        # Tempo input
        tempo_label = QLabel("Tempo (BPM, 0-239):")
        layout.addWidget(tempo_label)
        
        self.tempo_input = QSpinBox()
        self.tempo_input.setRange(0, 239)
        self.tempo_input.setValue(120)
        layout.addWidget(self.tempo_input)
        
        # Danceability input
        dance_label = QLabel("Danceability (0.0 to 1.0):")
        layout.addWidget(dance_label)
        
        self.dance_input = QDoubleSpinBox()
        self.dance_input.setRange(0.0, 1.0)
        self.dance_input.setSingleStep(0.1)
        self.dance_input.setValue(0.5)
        layout.addWidget(self.dance_input)
        
        # Filter button
        filter_btn = QPushButton("Apply Filters")
        filter_btn.clicked.connect(self.apply_filters)
        layout.addWidget(filter_btn)
        
        # Results list
        self.results_list = QListWidget()
        layout.addWidget(self.results_list)
        
        # Back button
        back_btn = QPushButton("Back to Home")
        back_btn.clicked.connect(lambda: self.parent.stacked_widget.setCurrentIndex(0))
        layout.addWidget(back_btn)
        
        self.setLayout(layout)
    
    def apply_filters(self):
        # Get selected tag
        selected_tag = self.tags_combo.currentText()
        
        # Call recommendation function
        recommendations = recommend_songs_by_genre_tempo_tags_danceability(
            song_genre=self.genre_combo.currentText(),
            song_tempo=str(self.tempo_input.value()),
            song_tags=selected_tag,
            danceability_value=str(self.dance_input.value()),
            year=str(self.year_input.value())
        )
        
        # Display results with custom widgets
        self.results_list.clear()
        for _, row in recommendations.iterrows():
            list_item = QListWidgetItem()
            song_widget = SongWidget(
                row['name'], 
                row['artist'], 
                row['spotify_preview_url'],
                row['duration']
            )
            list_item.setSizeHint(song_widget.sizeHint())
            self.results_list.addItem(list_item)
            self.results_list.setItemWidget(list_item, song_widget)

# Home Page
class HomePage(QWidget):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.parent = parent
        self.initUI()
        
    def initUI(self):
        layout = QVBoxLayout()
        
        # Search bar
        search_layout = QHBoxLayout()
        self.search_input = QLineEdit()
        self.search_input.setPlaceholderText("Enter song name...")
        self.search_input.returnPressed.connect(self.search_songs)
        search_layout.addWidget(self.search_input)
        
        search_btn = QPushButton("Search")
        search_btn.clicked.connect(self.search_songs)
        search_layout.addWidget(search_btn)
        
        filter_btn = QPushButton("Filters")
        filter_btn.clicked.connect(lambda: self.parent.stacked_widget.setCurrentIndex(2))
        search_layout.addWidget(filter_btn)
        
        login_btn = QPushButton("Login")
        login_btn.clicked.connect(lambda: self.parent.stacked_widget.setCurrentIndex(1))
        search_layout.addWidget(login_btn)
        
        layout.addLayout(search_layout)
        
        # Search results
        search_results_label = QLabel("Search Results:")
        search_results_label.setFont(QFont("Arial", 12, QFont.Bold))
        layout.addWidget(search_results_label)
        
        self.search_results_list = QListWidget()
        layout.addWidget(self.search_results_list)

        # Hybrid recommendations (initially hidden)
        self.hybrid_label = QLabel("Hybrid Recommendations (Content + Collaborative):")
        self.hybrid_label.setFont(QFont("Arial", 12, QFont.Bold))
        self.hybrid_label.hide()  # Initially hidden
        layout.addWidget(self.hybrid_label)
        
        self.hybrid_list = QListWidget()
        self.hybrid_list.hide()  # Initially hidden
        layout.addWidget(self.hybrid_list)
        
        # User-based recommendations
        user_rec_label = QLabel("Recommended for You:")
        user_rec_label.setFont(QFont("Arial", 12, QFont.Bold))
        layout.addWidget(user_rec_label)
        
        self.user_rec_list = QListWidget()
        layout.addWidget(self.user_rec_list)
        
        # Show login message if not logged in
        if not self.parent.current_user:
            self.user_rec_list.addItem("Please login first to see recommendations...")

        self.setLayout(layout)
    
    def search_songs(self):
        song_name = self.search_input.text().strip()
        if song_name:
            # Get content-based recommendations
            recommendations = recommend_songs_by_name(song_name)
            self.search_results_list.clear()
            for _, row in recommendations.iterrows():
                list_item = QListWidgetItem()
                song_widget = SongWidget(
                    row['name'], 
                    row['artist'], 
                    row['spotify_preview_url'],
                    row['duration']
                )
                list_item.setSizeHint(song_widget.sizeHint())
                self.search_results_list.addItem(list_item)
                self.search_results_list.setItemWidget(list_item, song_widget)
            
            # Show hybrid recommendations if user is logged in
            if self.parent.current_user:
                hybrid_recs = hybrid_recommendation(song_name, self.parent.current_user)
                self.hybrid_list.clear()
                for _, row in hybrid_recs.iterrows():
                    list_item = QListWidgetItem()
                    song_widget = SongWidget(
                        row['name'], 
                        row['artist'], 
                        row['spotify_preview_url'],
                        row['duration']
                    )
                    list_item.setSizeHint(song_widget.sizeHint())
                    self.hybrid_list.addItem(list_item)
                    self.hybrid_list.setItemWidget(list_item, song_widget)
                
                # Show hybrid section
                self.hybrid_label.show()
                self.hybrid_list.show()
            else:
                # Hide hybrid section if not logged in
                self.hybrid_label.hide()
                self.hybrid_list.hide()

# Main Window
class MusicRecommendationApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.current_user = None
        self.initUI()
        
    def initUI(self):
        self.setWindowTitle("Music Recommendation System")
        self.setGeometry(100, 100, 800, 600)
        
        # Central widget and main layout
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        
        # Stacked widget for different pages
        self.stacked_widget = QStackedWidget()
        
        # Create pages
        self.home_page = HomePage(self)
        self.login_page = LoginPage(self)
        self.filter_page = FilterPage(self)
        
        # Add pages to stacked widget
        self.stacked_widget.addWidget(self.home_page)
        self.stacked_widget.addWidget(self.login_page)
        self.stacked_widget.addWidget(self.filter_page)
        
        # Set main layout
        layout = QVBoxLayout()
        layout.addWidget(self.stacked_widget)
        central_widget.setLayout(layout)
        
        # Apply some styling
        self.apply_styles()
    
    def apply_styles(self):
        # Set application style
        self.setStyleSheet("""
            QMainWindow {
                background-color: #f5f5f5;
            }
            QLabel {
                color: #333;
                padding: 5px;
            }
            QPushButton {
                background-color: #4CAF50;
                color: white;
                border: none;
                padding: 8px 16px;
                border-radius: 4px;
                font-weight: bold;
            }
            QPushButton:hover {
                background-color: #45a049;
            }
            QLineEdit, QSpinBox, QDoubleSpinBox, QComboBox {
                padding: 8px;
                border: 1px solid #ccc;
                border-radius: 4px;
            }
            QListWidget {
                border: 1px solid #ccc;
                border-radius: 4px;
                background-color: white;
                alternate-background-color: #f2f2f2;
            }
            QScrollArea {
                border: 1px solid #ccc;
                border-radius: 4px;
            }
        """)
    
    def update_recommendations(self):
        # Update user recommendations if logged in
        if self.current_user:
            recommendations = recommend_by_user_id(self.current_user)
            self.home_page.user_rec_list.clear()
            for _, row in recommendations.iterrows():
                list_item = QListWidgetItem()
                song_widget = SongWidget(
                    row['name'], 
                    row['artist'], 
                    row['spotify_preview_url'],
                    row['duration']
                )
                list_item.setSizeHint(song_widget.sizeHint())
                self.home_page.user_rec_list.addItem(list_item)
                self.home_page.user_rec_list.setItemWidget(list_item, song_widget)

def main():
    app = QApplication(sys.argv)
    window = MusicRecommendationApp()
    window.show()
    sys.exit(app.exec_())

if __name__ == '__main__':
    main()

ValueError: All arrays must be of the same length